# Tree-of-Text

## Config

In [ ]:
CONFIG = {
    "dataset": "MLB",
    "mode": "test",
    "prompt": "Zero-shot",
    "shot": 0,
    "operation_history": "chain",
    "operation_argument": "one-step",
    "operation_pool": [
        "root",
        "select_table",
        "select_row",
        "select_col",
        "count",
        "sort",
        "filter",
        "write",
    ],
    "max_depth": 5,
    "max_degree": 5,
    "tree_traversal": "DFS",
    "table_format": "CSV",
    "table_index": False,
    "planning_model": "gpt-4o-mini",
    "planning_tokens": 100,
    "planning_temperature": 0.7,
    "write_model": "gpt-4o-mini",
    "write_tokens": 100,
    "write_temperature": 0.3,
    "generating_model": "gpt-4o-mini",
    "generating_tokens": 1000,
    "generating_temperature": 0.5,
    "extraction_prompt": "few-shot_2",
    "extraction_model": "gpt-4o-mini",
    "extraction_tokens": 1000,
    "extraction_temperature": 0.5,
    "stop": 1744,
}

## Generate Report

### Read Tables

- Input: dataframe for each match  
- Output: tables for each match

In [ ]:
import pandas as pd


def read_tables(index, mode="test"):
    tables = {}

    tables["game"] = pd.read_csv(
        f"../../dataset/data/{mode}/table/{index}/game.csv")
    tables["home_line"] = pd.read_csv(
        f"../../dataset/data/{mode}/table/{index}/home_line.csv")
    tables["vis_line"] = pd.read_csv(
        f"../../dataset/data/{mode}/table/{index}/vis_line.csv")
    tables["box_score"] = pd.read_csv(
        f"../../dataset/data/{mode}/table/{index}/box_score.csv")
    tables['play_by_play'] = pd.read_csv(f"../../dataset/data/{mode}/table/{index}/play_by_play.csv")


    return tables

### Tree-of-Text

- Input: tables, operation history, operation pool
- Output: report

In [ ]:
from tqdm.notebook import tqdm
import time


def tree_of_text(tables, operation_history, operation_pool, level):
    # Log for each match
    log = {}
    log['level'] = level
    log['tables'] = tables_to_texts(tables)
    log['operation_history'] = operation_history
    log['operation_pool'] = operation_pool
    
    start_time = time.time()
    
    # If level is the max depth, write the report
    if level >= CONFIG['max_depth']-1:
        # Update the operation history
        new_operation_history = operation_history.copy()
        new_operation_history.append("write()")

        # Update the operation pool
        new_operation_pool = operation_pool.copy()
        new_operation_pool.remove("write")

        # Write the report based on the tables
        new_tables = tables.copy()
        report, child_log = write(new_tables, new_operation_history, new_operation_pool, level+1)
        
        log['children'] = [child_log]
        log['report'] = report
        
        end_time = time.time()
        log['time'] = end_time - start_time
        
        return report, log
    
    # Content Planning
    operations_arguments, log['planning'] = content_planning(tables, operation_history, operation_pool)
    log['operations_arguments'] = operations_arguments
    
    # Operation Execution
    reports, log['children'] = operation_execution(tables, operation_history, operation_pool, level, operations_arguments)
    log["reports"] = reports
    
    # Content Generating
    # report = "\n".join(reports)
    new_report, log['generating'] = content_generating(reports, level)
    log['report'] = new_report
    
    end_time = time.time()
    log['time'] = end_time - start_time
    
    return new_report, log

#### Utils

##### Tables to Texts

- Input: dictionary of tables
- Output: dictionary of texts

In [ ]:
def tables_to_texts(tables):
    texts = {}
    
    for table in tables:
        if CONFIG["table_format"] == "CSV":
            texts[table] = tables[table].to_csv(index=CONFIG["table_index"], index_label="index")
        elif CONFIG["table_format"] == "HTML":
            texts[table] = tables[table].to_html(index=CONFIG["table_index"], index_label="index")
        elif CONFIG["table_format"] == "Markdown":
            texts[table] = tables[table].to_markdown(index=CONFIG["table_index"], index_label="index")
        elif CONFIG["table_format"] == "PIPE":
            texts[table] = tables[table].to_csv(sep='|', index=CONFIG["table_index"], index_label="index")
        else:
            raise ValueError(
                f"{CONFIG['table_format']} is not a valid CONFIG['table_format'].")
            
    return texts

##### Calculate cost

- Input: input_tokens, output_tokens, model
- Output: total_cost

In [ ]:
def calculate_cost(input_tokens, output_tokens, model):
    pricing = {
        'gpt-4o': {'input': 2.50, 'output': 10.00},
        'gpt-4o-mini': {'input': 0.150, 'output': 0.600},
    }

    input_cost = (input_tokens / 1000000) * pricing[model]['input']
    output_cost = (output_tokens / 1000000) * pricing[model]['output']

    total_cost = input_cost + output_cost

    return total_cost

##### Sum cost

- Input: dictionary
- Output: total_cost

In [ ]:
def sum_cost(d):
    total_cost = 0
    if isinstance(d, dict):
        for key, value in d.items():
            if key == "cost":
                total_cost += value
            else:
                total_cost += sum_cost(value)
    elif isinstance(d, list):
        for item in d:
            total_cost += sum_cost(item)
            
    return total_cost

#### Content Planning

- Input: tables, operation history, operation pool
- Output: operations & arguments

In [ ]:
def content_planning(tables, operation_history, operation_pool):
    planning_log = {}
    
    # Read the prompts for planning
    system_prompt, user_prompt = read_prompts_for_planning(tables, operation_history, operation_pool)
    
    planning_log['system_prompt'] = system_prompt
    planning_log['user_prompt'] = user_prompt
    
    # Generate the content for planning
    content, cost = LLM_for_planning(system_prompt, user_prompt)
    
    planning_log['content'] = content
    planning_log['cost'] = cost
    
    # Extract the operations and arguments from the content
    operations_arguments = extract_operations_arguments(content)
    
    return operations_arguments, planning_log

##### Read Prompts for Planning

- Input: none
- Output: system_prompt and user_prompt

In [ ]:
def read_prompts_for_planning(tables, operation_history, operation_pool):
    # Read the system prompt
    with open("prompt/planning/system.txt", 'r') as file:
        system_prompt = file.read()
    
    # Replace the max_degree, max_depth and table_format in the system prompt
    system_prompt = system_prompt.replace("{MAX_DEGREE}", str(CONFIG["max_degree"]))
    system_prompt = system_prompt.replace("{MAX_DEPTH}", str(CONFIG["max_depth"]))
    system_prompt = system_prompt.replace("{TABLE_FORMAT}", str(CONFIG["table_format"]))
    system_prompt = system_prompt.replace("{PLANNING_TOKENS}", str(CONFIG["planning_tokens"]))
    
    # Replace the operation_description in the system prompt
    operation_description_prompt = ""
    for operation in CONFIG["operation_pool"]:
        with open(f"prompt/planning/operation_pool/{operation}.txt", 'r') as file:
            operation_prompt = file.read()
            
            # Operation example
            # Replace the tables in the operation prompt
            # tables_prompt = ""
            # texts = tables_to_texts(tables)
            # for text in texts:
            #     tables_prompt += f"##### {text}" + "\n\n"
            #     tables_prompt += texts[text] + "\n\n"
                
            # operation_prompt = operation_prompt.replace("{TABLES}", tables_prompt)
            
            if operation == "write":
                # Write the report based on the input table
                # report, write_log = write(tables)
                
                operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "write()")
                
                # Replace the report in the operation prompt
                # operation_prompt = operation_prompt.replace("{REPORT}", report)
            else:
                # Update the tables
                if operation == "root":
                    # new_tables = root(tables)
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "root()")
                elif operation == "select_table":
                    # new_tables = select_table(tables, ["set_1"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "select_table(set_1)")
                elif operation == "select_row":
                    # new_tables = select_row(tables, ["0", "10:20", "-1"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "select_row(0, 10:20, -1)")
                elif operation == "select_col":
                    # new_tables = select_col(tables, ["winner_score", "loser_score", "getpoint_player", "ball_type"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "select_col(winner_score, loser_score, getpoint_player, ball_type)")
                elif operation == "count":
                    # new_tables = count(tables, ["getpoint_player", "ball_type"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "count(getpoint_player, ball_type)")
                elif operation == "sort":
                    # new_tables = sort(tables, ["count:descending", "getpoint_player:ascending"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "sort(count:descending, getpoint_player:ascending)")
                elif operation == "filter":
                    # new_tables = filter(tables, ["count >= 2", "winner_score >= 11"])
                    operation_prompt = operation_prompt.replace("{OPERATIONS_ARGUMENTS}", "filter(count >= 2, winner_score >= 11)")
                    
                # Replace the new_tables in the operation prompt
                # new_tables_prompt = ""
                # new_texts = tables_to_texts(new_tables)
                # for new_text in new_texts:
                #     new_tables_prompt += f"##### {new_text}" + "\n\n"
                #     new_tables_prompt += new_texts[new_text] + "\n\n"

                # operation_prompt = operation_prompt.replace("{NEW_TABLES}", new_tables_prompt)
            
            operation_description_prompt += operation_prompt + "\n\n"
            
    system_prompt = system_prompt.replace("{OPERATION_DESCRIPTION}", operation_description_prompt)

    # Read the user prompt
    with open("prompt/planning/user.txt", 'r') as file:
        user_prompt = file.read()
        
    # Replace the table in the user prompt
    tables_prompt = ""
    texts = tables_to_texts(tables)
    for text in texts:
        tables_prompt += f"### {text}" + "\n\n"
        tables_prompt += texts[text] + "\n\n"
        
    user_prompt = user_prompt.replace("{TABLES}", tables_prompt)
        
    # Replace the operation_history in the user prompt
    operation_history_prompt = ""
    for operation in operation_history:
        operation_history_prompt += f"{operation} -> "
        
    user_prompt = user_prompt.replace("{OPERATION_HISTORY}", operation_history_prompt)
    
    # Replace the operation_pool in the user prompt
    operation_pool_prompt = str(operation_pool)
    user_prompt = user_prompt.replace("{OPERATION_POOL}", operation_pool_prompt)
    
    return system_prompt, user_prompt

##### LLM for planning

- Input: system_prompt, user_prompt
- Output: content, cost

In [ ]:
from openai import OpenAI


def LLM_for_planning(system_prompt, user_prompt):
    client = OpenAI()

    response = client.chat.completions.create(
        model=CONFIG["planning_model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=CONFIG["planning_tokens"],
        temperature=CONFIG["planning_temperature"],
    )

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = calculate_cost(input_tokens, output_tokens, CONFIG["planning_model"])

    content = response.choices[0].message.content
    
    return content, cost

##### Extract Operations

- Input: content from LLM
- Output: extracted operations

In [ ]:
import re


def extract_operations_arguments(content):
    # Regular expression pattern to match each operation
    pattern = r'\w+\(.*?\)'

    # Find all matches
    operations_arguments = re.findall(pattern, content)
    
    return operations_arguments

#### Operation Execution

- Input: tables, operation_history, operation_pool, level, operations_arguments
- Output: reports, child_logs

In [ ]:
def operation_execution(tables, operation_history, operation_pool, level, operations_arguments):
    child_logs = []
    reports = []
    
    for operation_argument in tqdm(operations_arguments[:CONFIG['max_degree']], desc=f"level {level+1}", position=level+1):
        operation = operation_argument.split('(')[0]
        arguments = operation_argument.split('(')[1].split(')')[0].split(', ')

        # Check the operation is in the operation pool
        if operation not in operation_pool:
            child_log = {
                "error": f"{operation} is not in the operation pool."
            }
            child_logs.append(child_log)
            continue

        # Update the operation history
        new_operation_history = operation_history.copy()
        new_operation_history.append(operation_argument)

        # Update the operation pool
        new_operation_pool = operation_pool.copy()
        new_operation_pool.remove(operation)

        # Check the operation
        if operation == "write":
            # Write the report based on the tables
            new_tables = tables.copy()
            report, child_log = write(new_tables, new_operation_history, new_operation_pool, level+1)
        else:
            # Update the tables
            if operation == "root":
                new_tables = root(tables)
            elif operation == "select_table":
                new_tables = select_table(tables, arguments)
            elif operation == "select_row":
                new_tables = select_row(tables, arguments)
            elif operation == "select_col":
                new_tables = select_col(tables, arguments)
            elif operation == "count":
                new_tables = count(tables, arguments)
            elif operation == "sort":
                new_tables = sort(tables, arguments)
            elif operation == "filter":
                new_tables = filter(tables, arguments)

            # Go to next layer of the tree
            report, child_log = tree_of_text(new_tables, new_operation_history, new_operation_pool, level+1)

        # Append the report to the reports
        reports.append(report)

        # Save child_log
        child_logs.append(child_log)
        
    return reports, child_logs

##### root

- Input: tables
- Output: same tables

In [ ]:
def root(tables):
    new_tables = tables.copy()
    
    return new_tables

##### select_table

- Input: tables, table names
- Output: selected tables

In [ ]:
def select_table(tables, arguments):
    new_tables = {}
    
    for table in tables:
        if table in arguments:
            new_tables[table] = tables[table]
    
    return new_tables

##### select_row

- Input: tables, row indices
- Output: tables with selected rows

In [ ]:
def select_row(tables, arguments):
    new_tables = {}

    for table in tables:
        selected_rows = []
        
        for argument in arguments:
            try:
                if ':' in argument:
                    i, j = argument.split(':')
                    selected_rows.append(tables[table].iloc[int(i):int(j)+1])
                else:
                    i = argument
                    selected_rows.append(tables[table].iloc[[int(i)]])
            except:
                # print(f"[ERROR] select_row({table}, {argument}) is not valid!")
                pass
        
        try:
            new_tables[table] = pd.concat(selected_rows, ignore_index=True)
        except:
            new_tables[table] = tables[table]
            print(f"[ERROR] select_row({table}, {arguments}) is not valid!")

    return new_tables

##### select_col

- Input: tables, column names
- Output: tables with selected columns

In [ ]:
def select_col(tables, arguments):
    new_tables = {}

    for table in tables:
        selected_cols = []

        for argument in arguments:
            try:
                selected_cols.append(tables[table][argument])
            except:
                # print(f"[ERROR] select_col({table}, {argument}) is not valid!")
                pass

        try:
            new_tables[table] = pd.concat(selected_cols, axis=1)
        except:
            new_tables[table] = tables[table]
            print(f"[ERROR] select_col({table}, {arguments}) is not valid!")

    return new_tables

##### count

- Input: tables, column names
- Output: new tables with the column names and count.

In [ ]:
def count(tables, arguments):
    new_tables = {}

    for table in tables:
        count_arguments = []
        
        for argument in arguments:
            if argument in tables[table].columns:
                count_arguments.append(argument)
                
        try:
            # Group by multiple columns and count unique values
            new_tables[table] = tables[table].groupby(count_arguments).size().reset_index(name='count')
        except:
            new_tables[table] = tables[table]
            print(f"[ERROR] count({table}, {arguments}) is not valid!")

    return new_tables

##### sort

- Input: tables, column names:sorting order
- Output: sorted tables

In [ ]:
def sort(tables, arguments):
    new_tables = {}

    for table in tables:
        sort_columns = []
        sort_orders = []

        for argument in arguments:
            sort_column = argument.split(':')[0]
            sort_order = argument.split(':')[1]
            
            if sort_column in tables[table].columns:
                sort_columns.append(sort_column)
                if sort_order == "ascending":
                    sort_orders.append(True)
                elif sort_order == "descending":
                    sort_orders.append(False)
                else:
                    print(f"[ERROR] sort({table}, {arguments}) is not valid!")

        try:
            # Sort by multiple columns and orders
            new_tables[table] = tables[table].sort_values(by=sort_columns, ascending=sort_orders)
        except:
            new_tables[table] = tables[table]
            print(f"[ERROR] sort({table}, {arguments}) is not valid!")

    return new_tables

##### filter

- Input: tables, column names and conditions
- Output: filtered tables

In [ ]:
def filter(tables, arguments):
    new_tables = tables.copy()

    for table in tables:
        for argument in arguments:
            try:
                filter_column = argument.split(' ')[0]
                filter_symbol = argument.split(' ')[1]
                filter_value = " ".join(argument.split(' ')[2:])
                try:
                    filter_value = int(filter_value)
                except:
                    filter_value = str(filter_value)
                        
                try:
                    if filter_symbol == "==":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] == filter_value].reset_index(drop=True)
                    elif filter_symbol == "!=":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] != filter_value].reset_index(drop=True)
                    elif filter_symbol == ">":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] > filter_value].reset_index(drop=True)
                    elif filter_symbol == ">=":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] >= filter_value].reset_index(drop=True)
                    elif filter_symbol == "<":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] < filter_value].reset_index(drop=True)
                    elif filter_symbol == "<=":
                        new_tables[table] = new_tables[table][new_tables[table][filter_column] <= filter_value].reset_index(drop=True)
                    else:
                        print(f"[ERROR] filter({table}, {filter_symbol}) is not valid!")
                except:
                    print(f"[ERROR] filter({table}, {filter_column}) is not valid!")
            except:
                print(f"[ERROR] filter({table}, {argument}) is not valid!")

    return new_tables

##### write

- Input: tables
- Output: report

In [ ]:
def write(tables, operation_history, operation_pool, level):
    write_log = {}
    write_log['level'] = level
    write_log['tables'] = tables_to_texts(tables)
    write_log['operation_history'] = operation_history
    write_log['operation_pool'] = operation_pool
    
    start_time = time.time()
    
    # Read the system prompt
    with open("prompt/write/system.txt", 'r') as file:
        system_prompt = file.read()
    
    # Replace the table_format and write_tokens in the system prompt
    system_prompt = system_prompt.replace("{TABLE_FORMAT}", str(CONFIG["table_format"]))
    system_prompt = system_prompt.replace("{WRITE_TOKENS}", str(CONFIG["write_tokens"]))
    write_log['system_prompt'] = system_prompt
    
    # Read the user prompt
    with open("prompt/write/user.txt", 'r') as file:
        user_prompt = file.read()
    
    # Replace the table in the user prompt
    tables_prompt = ""
    texts = tables_to_texts(tables)
    for text in texts:
        tables_prompt += f"### {text}" + "\n\n"
        tables_prompt += texts[text] + "\n\n"

    user_prompt = user_prompt.replace("{TABLES}", tables_prompt)
    write_log['user_prompt'] = user_prompt
    
    # Write the report based on the input table
    client = OpenAI()

    response = client.chat.completions.create(
        model=CONFIG["write_model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=CONFIG["write_tokens"],
        temperature=CONFIG["write_temperature"],
    )

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = calculate_cost(input_tokens, output_tokens, CONFIG["write_model"])
    write_log['cost'] = cost

    content = response.choices[0].message.content
    
    # Postprocess the report
    report = content
    write_log['report'] = report
    
    end_time = time.time()
    write_log['time'] = end_time - start_time
    
    return report, write_log

#### Content Generating

- Input: reports, level
- Output: new_report

In [ ]:
def content_generating(reports, level):
    generating_log = {}
    
    if level == 1:
        if len(reports) == 1:
            return reports[0], generating_log
        else:
            # Read the prompts for generating
            system_prompt, user_prompt = read_prompts_for_generating(reports)
            
            generating_log['system_prompt'] = system_prompt
            generating_log['user_prompt'] = user_prompt

            # Generate the new_report for generating
            new_report, cost = LLM_for_generating(system_prompt, user_prompt)

            generating_log['cost'] = cost

            return new_report, generating_log
    else:
        if len(reports) == 1:
            return reports[0], generating_log
        else:
            # Concat the reports
            new_report = "\n".join(reports)
            return new_report, generating_log

##### Read Prompts for Generating

- Input: reports
- Output: system_prompt, user_prompt

In [ ]:
def read_prompts_for_generating(reports):
    # Read the system prompt
    with open("prompt/generating/system.txt", 'r') as file:
        system_prompt = file.read()
        
    # Replace the generating_tokens in the system prompt
    system_prompt = system_prompt.replace("{GENERATING_TOKENS}", str(CONFIG["generating_tokens"]))
    
    # Read the user prompt
    with open("prompt/generating/user.txt", 'r') as file:
        user_prompt = file.read()
        
    # Replace the reports in the user prompt
    reports_prompt = ""
    for i, report in enumerate(reports):
        reports_prompt += f"### Report {i+1}" + "\n\n"
        reports_prompt += report + "\n\n"
        
    user_prompt = user_prompt.replace("{REPORTS}", reports_prompt)
    
    return system_prompt, user_prompt

##### LLM for Generating

- Input: system_prompt, user_prompt
- Output: new_report, cost

In [ ]:
from openai import OpenAI


def LLM_for_generating(system_prompt, user_prompt):
    client = OpenAI()

    response = client.chat.completions.create(
        model=CONFIG["generating_model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=CONFIG["generating_tokens"],
        temperature=CONFIG["generating_temperature"],
    )

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = calculate_cost(input_tokens, output_tokens, CONFIG["generating_model"])

    new_report = response.choices[0].message.content

    return new_report, cost

### Save Report

- Input: match, report
- Output: none

In [ ]:
import os


def save_report(index, report):
    os.makedirs(f'report/{index}/', exist_ok=True)
    with open(f"report/{index}/report.txt", 'w') as file:
        file.write(report)

### Save Log

- Input: match, log
- Output: none

In [ ]:
import os
import json


def save_log(index, log):
    os.makedirs(f'log/{index}/', exist_ok=True)
    with open(f"log/{index}/log.json", 'w') as file:
        json.dump(log, file, indent=4)

### Run

In [ ]:
import json
from tqdm.notebook import tqdm
import time
import os


print("> Start Generate Report!")

# Save the config
print("> Save the config...")
os.makedirs('config/', exist_ok=True)
with open('config/config.json', 'w') as file:
    json.dump(CONFIG, file, indent=4)
    
print("CONFIG:", CONFIG)

dirs = os.listdir(f"../../dataset/data/{CONFIG['mode']}/table/")
stop = min(CONFIG["stop"], len(dirs))

for index in tqdm(range(stop), desc=str(CONFIG['mode']), position=0):
    start_time = time.time()
    
    # Read the tables
    print(">> Read the tables...")
    tables = read_tables(index, CONFIG['mode'])
    
    # Init the operation history
    init_operation_history = ["root()"]

    # Init the operation pool
    init_operation_pool = CONFIG['operation_pool'].copy()
    init_operation_pool.remove("root")
    
    # Init the level
    level = 0
    
    # Tree-of-Report
    print(">> Tree-of-Report...")
    for _ in tqdm(init_operation_history, desc=f"level {level+1}", position=level+1):
        report, log = tree_of_text(tables, init_operation_history, init_operation_pool, level+1)
        
    # Calculate the total time
    print(">> Calculate the total time...")
    end_time = time.time()
    total_time = end_time - start_time
    print(f"Total time: {total_time}s")
    log['total_time'] = total_time
    
    # Calculate the total cost
    print(">> Calculate the total cost...")
    total_cost = sum_cost(log)
    print(f"Total cost: {total_cost}$")
    log['total_cost'] = total_cost
    
    # Save the report
    print(">> Save the report...")
    save_report(index, report)
    
    # Save the log
    print(">> Save the log...")
    save_log(index, log)
    
    # break
    
print("> End Generate Report!")

## Relation Extraction

### Read Report

- Input: match
- Output: report for this match

In [ ]:
def read_report(index):
    with open(f"report/{index}/report.txt", 'r') as file:
        report = file.read()
    
    return report

### Extract Relations

- Input: text
- Output: relations

In [ ]:
from openai import OpenAI
import re


def extract_relations(index, report):
    relations = []
    
    # Read the system prompt
    with open(f"../../dataset/relation_extraction/prompt/{CONFIG['extraction_prompt']}/system.txt", "r") as file:
        system_prompt = file.read()
        
    # Read the user prompt
    with open(f"../../dataset/relation_extraction/prompt/{CONFIG['extraction_prompt']}/user.txt", "r") as file:
        user_prompt = file.read()
    
    user_prompt = user_prompt.replace("{REPORT}", report)
    
    # Read the table relation
    with open(f"../../dataset/relation_extraction/relation/{CONFIG['extraction_prompt']}/{CONFIG['mode']}/{index}/table.json", "r") as file:
        table_relation = json.load(file)
    
    user_prompt = user_prompt.replace("{TABLE_RELATION}", str(table_relation))
    
    # LLM relation extractor
    client = OpenAI()

    response = client.chat.completions.create(
        model=CONFIG["extraction_model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=CONFIG["extraction_tokens"],
        temperature=CONFIG["extraction_temperature"],
    )

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_cost = calculate_cost(input_tokens, output_tokens, CONFIG["extraction_model"])

    content = response.choices[0].message.content
    
    # print("cost:", cost)
    # print("content:", content)
    
    # Compile regex pattern to match (*|*|*)
    pattern = re.compile(r"\(.*?\|.*?\|.*?\)")

    # Search for all matches in the text
    relations = pattern.findall(content)
    
    # print("relations:", relations)
    
    return relations, total_cost

In [ ]:
def calculate_cost(input_tokens, output_tokens, model):
    pricing = {
        'gpt-4o': {'input': 2.50, 'output': 10.00},
        'gpt-4o-mini': {'input': 0.150, 'output': 0.600},
    }

    input_cost = (input_tokens / 1000000) * pricing[model]['input']
    output_cost = (output_tokens / 1000000) * pricing[model]['output']

    total_cost = input_cost + output_cost

    return total_cost

### Save Relations

- Input: match, relations
- Output: none

In [ ]:
import os
import json


def save_relations(index, relations):
    os.makedirs(f"relation/{index}/", exist_ok=True)
    with open(f"relation/{index}/report.json", "w") as file:
        json.dump(relations, file, indent=4)

### Run

In [ ]:
import json
from tqdm.notebook import tqdm
import time


print("> Start Relation Extraction!")
    
print("CONFIG:", CONFIG)

dirs = os.listdir(f"report/")
stop = min(CONFIG["stop"], len(dirs))

for index in tqdm(range(stop), desc=str(CONFIG['mode']), position=0):
    # Read the report
    # print(">> Read the report...")
    report = read_report(index)
    
    # Extract relations
    # print(">> Extract relations...")
    relations, cost = extract_relations(index, report)
    
    # Save the relations
    # print(">> Save the relations...")
    save_relations(index, relations)
    
    # break
    
print("> End Relation Extraction!")

## Evaluate Report

### Read Text

- Input: match
- Output: text for this match

In [ ]:
import os


def read_text(index):
    with open(f"../../dataset/data/{CONFIG['mode']}/text/{index}/text.txt", 'r') as file:
        text = file.read()

    # os.makedirs(f'text/{match["video"]}/', exist_ok=True)
    # with open(f"text/{match['video']}/text.txt", 'w') as file:
    #     file.write(text)

    return text

### Read Report

- Input: match
- Output: report for this match

In [ ]:
def read_report(index):
    with open(f"report/{index}/report.txt", 'r') as file:
        report = file.read()
    
    return report

### Read Log

- Input: match
- Output: log for this match

In [ ]:
def read_log(index):
    with open(f"log/{index}/log.json", 'r') as file:
        log = json.load(file)
    
    return log

### Read Relation

- Input: match
- Output: table_relations, text_relations, report_relations

In [ ]:
def read_relation(index):
    with open(f"../../dataset/relation_extraction/relation/{CONFIG['extraction_prompt']}/{CONFIG['mode']}/{index}/table.json", "r") as file:
        table_relations = json.load(file)
        
    with open(f"relation/{index}/table.json", 'w') as file:
        json.dump(table_relations, file, indent=4)

    with open(f"../../dataset/relation_extraction/relation/{CONFIG['extraction_prompt']}/{CONFIG['mode']}/{index}/text.json", "r") as file:
        text_relations = json.load(file)
        
    with open(f"relation/{index}/text.json", 'w') as file:
        json.dump(text_relations, file, indent=4)

    with open(f"relation/{index}/report.json", 'r') as file:
        report_relations = json.load(file)
    
    return table_relations, text_relations, report_relations

### Relation Generation

- Input: table_relations, report_relations
- Output: count, precision

In [ ]:
def relation_generation(table_relations, report_relations):
    # Calculate the count of relations in each file
    table_count = len(table_relations)
    report_count = len(report_relations)

    # Calculate the number of correct relations (relations present in both files)
    correct_relations = set(table_relations).intersection(set(report_relations))
    correct_count = len(correct_relations)

    # Calculate the precision
    precision = correct_count / report_count if report_count > 0 else 0
    
    return correct_count, precision

### Content Selection

- Input: text_relations, report_relations
- Ouput: precision, recall, F-measure

In [ ]:
def content_selection(text_relations, report_relations):
    # Calculate the count of relations in each file
    text_count = len(text_relations)
    report_count = len(report_relations)

    # Calculate the number of correct relations (relations present in both files)
    correct_relations = set(text_relations).intersection(set(report_relations))
    correct_count = len(correct_relations)

    # Calculate precision
    precision = correct_count / report_count if report_count > 0 else 0

    # Calculate recall
    recall = correct_count / text_count if text_count > 0 else 0

    # Calculate F-measure (F1 score)
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
    
    return precision, recall, f1

### Content Ordering

- Input: text_relations, report_relations
- Ouput: Damerau-Levenshtein Distance

In [ ]:
import textdistance


def content_ordering(text_relations, report_relations):
    # Calculate the raw Damerau-Levenshtein distance
    dld = textdistance.damerau_levenshtein.distance(text_relations, report_relations)
    
    # Normalize by dividing by the maximum length of the two strings
    max_length = max(len(text_relations), len(report_relations))
    if max_length == 0:
        return 1.0  # Both strings are empty, considered identical

    normalized_dld = dld / max_length

    # Complement of normalized DLD
    complement_dld = 1 - normalized_dld

    return complement_dld

### BLEU

- Input: text, report
- Output: bleu

In [ ]:
from nltk.translate.bleu_score import sentence_bleu


def calculate_bleu(text, report):
    texts = [text.split()]
    reports = report.split()
    
    bleu = sentence_bleu(texts, reports)

    return bleu

### ROUGE

- Input: text, report
- Output: rouge

In [ ]:
from rouge_score import rouge_scorer


def calculate_rouge(text, report):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    score = scorer.score(text, report)
    
    rouge = {
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure,
    }

    return rouge

### BERTScore

- Input: text, report
- Output: BERTScore

In [ ]:
from bert_score import score
    
    
def calculate_bertscore(text, report):
    P, R, F1 = score([report], [text], lang="en",verbose=False, device='cuda')
    
    bertscore = {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

    return bertscore

### Calculate Sum & Average

- Input: evaluations
- Output: sum and average of evaluations

In [ ]:
def calculate_sum_average(evaluations):
    sum_evaluation = {}
    average_evaluation = {}
    
    for video, evaluation in evaluations.items():
        for key, value in evaluation.items():
            if key not in sum_evaluation:
                sum_evaluation[key] = value
            else:
                sum_evaluation[key] += value
                
    for key, value in sum_evaluation.items():
        average_evaluation[key] = value / len(evaluations)
    
    return sum_evaluation, average_evaluation

### Save Evaluations

- Input: evaluations
- Output: none

In [ ]:
import json
import pandas as pd


def save_evaluations(evaluations):
    os.makedirs(f'evaluation/', exist_ok=True)
    
    with open("evaluation/evaluations.json", 'w') as file:
        json.dump(evaluations, file, indent=4)
    
    df = pd.DataFrame(evaluations).transpose()
    df.to_csv("evaluation/evaluations.csv")

### Run

In [ ]:
import json
from tqdm.notebook import tqdm
import pandas as pd


print("> Start Evaluate Report!")

print("CONFIG:", CONFIG)

evaluations = {}

dirs = os.listdir(f"report/")
stop = min(CONFIG["stop"], len(dirs))

for index in tqdm(range(stop)):
    evaluation = {}
    
    # Read the text
    # print(">> Read the text...")
    text = read_text(index)
    # print("text:")
    # print(text)
    
    # Read the report
    # print(">> Read the report...")
    report = read_report(index)
    # print("report:")
    # print(report)

    # Read the log
    # print(">> Read the log...")
    log = read_log(index)
    # print("log:")
    # print(log)
    
    # Read the relation
    # print(">> Read the relation...")
    table_relations, text_relations, report_relations = read_relation(index)
        
    # Calculate the Relation Generation
    correct_count, precision = relation_generation(table_relations, report_relations)
    evaluation['RG_#'] = correct_count
    evaluation['RG_P'] = precision
    
    # Calculate the Content Selection
    precision, recall, f1 = content_selection(text_relations, report_relations)
    evaluation['CS_P'] = precision
    evaluation['CS_R'] = recall
    evaluation['CS_F'] = f1
    
    # Calculate the Content Ordering
    dld = content_ordering(text_relations, report_relations)
    evaluation['CO_DLD'] = dld
    
    # Calculate the BLEU score
    # print(">> Calculate the BLEU score...")
    bleu = calculate_bleu(text, report)
    evaluation['bleu'] = bleu
    # print("bleu:")
    # print(bleu)
    
    # Calculate the ROUGE score
    # print(">> Calculate the ROUGE score...")
    rouge = calculate_rouge(text, report)
    evaluation['rouge1'] = rouge['rouge1']
    evaluation['rouge2'] = rouge['rouge2']
    evaluation['rougeL'] = rouge['rougeL']
    # print("rouge:")
    # print(rouge)
    
    # Calculate the BERTScore
    # print(">> Calculate the BERTScore...")
    bertscore = calculate_bertscore(text, report)
    evaluation['bertscore_precision'] = bertscore['precision']
    evaluation['bertscore_recall'] = bertscore['recall']
    evaluation['bertscore_f1'] = bertscore['f1']
    # print("bertscore:")
    # print(bertscore)
    
    # Calculate the total time
    # print(">> Calculate the total time...")
    evaluation['total_time'] = log['total_time']
    # print("total_time:")
    # print(log['total_time'])
    
    # Calculate the total cost
    # print(">> Calculate the total cost...")
    evaluation['total_cost'] = log['total_cost']
    # print("total_cost:")
    # print(log['total_cost'])
    
    # Save the evaluation
    # print(">> Save the evaluation...")
    # save_evaluation(match, evaluation)
    
    evaluations[str(index)] = evaluation
    
    # break

# Calculate the sum and average evaluation
print("> Calculate the average evaluation...")
sum_evaluation, average_evaluation = calculate_sum_average(evaluations)
evaluations['sum'] = sum_evaluation
evaluations['average'] = average_evaluation
    
# Save the evaluations
print("> Save the evaluations...")
save_evaluations(evaluations)

print("> End Evaluate Report!")